In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import edunn as nn

# `Sequential` Model for Neural Networks


We have already implemented layers/models of every kind: dense layers, activation functions, error functions, and so on. On top of that, we have initializers, an optimizer based on stochastic gradient descent, and models that combine other layers, such as `LinearRegression` and `LogisticRegression`. 

To take the next step and be able to define simple neural networks, we are going to implement the `Sequential` model. This model generalizes the ideas applied in `LinearRegression`, `LogisticRegression` and `Dense`, namely, building a layer out of other layers. In those earlier cases, the layers to be used were predefined. `Sequential` will let us use any combination of layers we want.



# Creating a `Sequential` model


A `Sequential` model must be created with a list of other models/layers. That way, we will specify which transformations will be applied, and in what order, to produce the network's output.

We can see several examples in which we create a linear regression model, a logistic regression model, or a Dense layer on top of the `Sequential` model.

`Sequential` also has a very handy method, `summary()`, which lets us obtain a description of the layers and their parameters.


In [ ]:
din=5
dout=3

# We create a linear regression model 
layers = [nn.Linear(din,dout), nn.Bias(dout)]
linear_regression = nn.Sequential(layers,name="Linear Regression")
print(linear_regression.summary())


# We create a linear regression model, but without the auxiliary variable `layers`
linear_regression = nn.Sequential([nn.Linear(din,dout),
                       nn.Bias(dout),
                      ],name="Linear Regression (no auxiliary variable)")
print(linear_regression.summary())

# We create a logistic regression model 
logistic_regression = nn.Sequential([nn.Linear(din,dout),
                       nn.Bias(dout),
                       nn.Softmax()
                      ],name="Logistic Regression")
print(logistic_regression.summary())


# We create a Dense-layer-style model with ReLU activation
dense_relu = nn.Sequential([nn.Linear(din,dout),
                       nn.Bias(dout),
                       nn.ReLU()
                      ],name="Dense-style layer with ReLU activation")
print(dense_relu.summary())




# Multi-layer networks with `Sequential`

We are also going to build our first multi-layer neural networks, simply by adding more layers to the model.

In [ ]:

# We create a network with two Dense layers, and an internal dimensionality of 3
network_layer2 = nn.Sequential([nn.Dense(din,3,"relu"),
                               nn.Dense(3,dout,"id")
                      ],name="Two-layer network")
print(network_layer2.summary())



# We create a network with 4 Dense layers
# internal dimensions of 2, 4 and 3
# and a final softmax activation function
network_layer4 = nn.Sequential([nn.Dense(din,2,"relu"),
                               nn.Dense(2,4,"tanh"),
                               nn.Dense(4,3,"sigmoid"),
                               nn.Dense(3,dout,"softmax"),
                      ],name="Four-layer network")
print(network_layer4.summary())

# `Sequential` parameters

The `Sequential` model also makes it easy to obtain the parameters of all its inner models. For that we have already implemented the `get_parameters` method, which lets us obtain _all_ the parameters of the inner models, but renamed so that, if two models happen to use the same name for their parameters, those names do not clash.


In [ ]:
print("Parameter names of network_layer2")
print(network_layer2.get_parameters().keys())

print("Parameter names of network_layer4")
print(network_layer4.get_parameters().keys())

# `forward` method of `Sequential`


We are now going to implement the `forward` method of `Sequential`. To do so, given an input `x` and a sequence of models `M_1,M_2,...,M_n` inside `Sequential`, we must compute the output `y` as:

$$ y = M_n(...(M_2(M_1(x))...)$$

In terms of code, we must iterate over the models (starting with the first one) and apply the `forward` method

````python
for m in models:
    x = m.forward(x)
return x
````

Implement `forward` for the `Sequential` class in `edunn/models/sequential.py`. 

In [ ]:
x = np.array([[3,-7],
             [-3,7]])

w = np.array([[2, 3, 4],[4,5,6]])
b = np.array([1,2,3])
linear_initializer = nn.initializers.Constant(w)
bias_initializer = nn.initializers.Constant(b)
layer=nn.Sequential([nn.Linear(2,3,initializer=linear_initializer),
                     nn.Bias(3,initializer=bias_initializer)
                    ])
y = np.array([[-21, -24, -27],
              [ 23, 28,  33]])

nn.utils.check_same(y,layer.forward(x))

linear_initializer = nn.initializers.Constant(-w)
bias_initializer = nn.initializers.Constant(-b)
layer=nn.Sequential([nn.Linear(2,3,initializer=linear_initializer),
                     nn.Bias(3,initializer=bias_initializer)
                    ])
nn.utils.check_same(-y,layer.forward(x))

# `backward` method


Just as with `Dense`, to implement `backward` you will also have to call the `backward` of each of the models in the _reverse_ order of the forward pass. Given a tensor `δEδy` that contains the derivatives of the error with respect to each value of the output `y`, we must compute:
* `δEδx`, the derivative of the error with respect to the input `x`
* `δEδp_i`, the derivative of the error with respect to each parameter `p_i`

To do this, we must iterate over the models (starting with the last one) and apply the `backward` method, propagating the error backwards and collecting along the way the most important part: the derivatives of the error with respect to the parameters. In terms of code,

````python
δEδp = {}
for m_i in reverse(models):
    δEδy, δEδp_i = m_i.backward(δEδy)
    add the gradients of δEδp_i to δEδp
return δEδy,δEδp
````
In this case we also help you out with the `merge_gradients` function, which you can call as `self.merge_gradients(layer,δEδp,gradients)`. This function lets you add the `δEδp` parameters of the layer `layer` to the final gradients dictionary `gradients` that must be returned.


In [ ]:
samples = 100
batch_size=2
features_in=3
features_out=5
input_shape=(batch_size,features_in)

# Test derivatives of a Sequential model with random values for `w`
layer=nn.Sequential([nn.Linear(features_in,features_out),
                     nn.Bias(features_out),
                     nn.ReLU()
                    ])
nn.utils.check_gradient.common_layer(layer,input_shape,samples=samples)    


# Congratulations! 

You have implemented all the basic functionality of a neural network library!

Now we are going to define some neural networks in order to improve performance over the linear models (Linear Regression and Logistic Regression)
